In [1]:
from datetime import datetime
from pathlib import Path

import fire
import numpy as np
import numpy.typing as npt
import torch
from jsonlines import jsonlines
from loguru import logger
from torch import Tensor, nn
from torch.utils.data import DataLoader, TensorDataset

from fourier_supervised_cleanroom import mk_train_test, sign_signal
from fourier_supervised_cleanroom_2023_09_27 import get_lattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from misc_utils import keep_serializable, make_unpacked_configurations, hadamard_transform
from parity import parity
from spin_lattices import KagomeLattice, SpinLattice, SquareLattice, TriangularLattice
from nn_supervised_reproduction import default_config, sign_overlap, accuracy, train, SignDenseNetXor
from fourier_supervised_cleanroom import fit_fourier_series
from parity import popcount
from typing import Any

In [2]:
config = default_config | {"eps_train": [0.01], "J2s": [1.0], "n_test": 50000}
config

{'J2s': [1.0],
 'eps_train': [0.01],
 'n_test': 50000,
 'sampling_power_train': 2.0,
 'architecture': 'dense',
 'n_hidden': 512,
 'hidden_layers': 1,
 'epochs': 1000,
 'write_each_epoch': 1,
 'lr': 0.001,
 'batch_size': 64,
 'shuffle': True}

In [18]:
config |= {'architecture': 'dense+xor', 'xor_strategy': sample_xors_uniform(), 'n_xors': 1}

In [22]:
expansion = fit_fourier_series(system.canonical_basis.states, signal_fn, system.number_spins)

In [23]:
probs = expansion ** 2
probs /= probs.sum()

In [27]:
(popcount(np.arange(2 ** system.number_spins, dtype='uint64')) * probs).sum()

11.999999999999996

In [20]:
lattice = KagomeLattice(2, 4)
J2 = 1
start_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# (output_dir / str(task_id)).mkdir(parents=True, exist_ok=True)
out = []

xor_strategy = sample_xors_uniform()
n_xors = 1

for J2 in config["J2s"]:
    #    logger.debug(f"Running {task_id=} {J2=}. Creating system...")
    system = HeisenbergJ1J2(
        lattice=lattice, J1=1, J2=J2, use_symmetries=False, spin_inversion=None
    )
    system.get_eigenstates(1)
    signal_fn = sign_signal(system)
    sign_overlap_fn = sign_overlap(system)
    accuracy_fn = accuracy(system)

    for eps_train in config["eps_train"]:
        logger.debug(f"{eps_train=}. Making train and test states...")
        n_train = int(system.canonical_basis.states.shape[0] * eps_train)
        n_test = config["n_test"]
        train_states, test_states = mk_train_test(
            system,
            n_train=n_train,
            n_test=n_test,
            sampling_power_train=config["sampling_power_train"],
        )
        net = get_network(config, system)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"])

        # Create a TensorDataset from your inputs X and Y
        dataset = TensorDataset(
            torch.from_numpy(train_states.astype(np.int64)),
            torch.from_numpy(signal_fn(train_states) == -1).to(torch.long),
        )

        dataloader = DataLoader(
            dataset, batch_size=config["batch_size"], shuffle=config["shuffle"]
        )

        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        net.to(device)

        for epoch in range(config["epochs"]):
            train_loss = train(net, dataloader, criterion, optimizer, device)
            if epoch % config["write_each_epoch"] == 0:
                current_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                logger.info(f"Epoch {epoch}, train loss: {train_loss:.4f}")
                test_overlap = sign_overlap_fn(test_states, net, device)
                test_accuracy = accuracy_fn(test_states, net, device)
                logger.info(f"Overlap: {test_overlap:.4f}")
                # with jsonlines.open(
                #     output_dir / str(task_id) / f"results.jsonl", mode="a"
                # ) as writer:
                #     writer.write(
                out.append(
                    keep_serializable(config)
                    | {
                        "test_overlap": test_overlap,
                        "test_accuracy": test_accuracy,
                        "train_loss": train_loss,
                        "epoch": epoch,
                        "J2": J2,
                        "eps_train": eps_train,
                        "start_timestamp": start_timestamp,
                        "current_timestamp": current_timestamp,
                    }
                )

2023-10-18 16:49:13.569 | DEBUG    | heisenberg_hamiltonians:__init__:472 - number_spins=24
2023-10-18 16:49:13.570 | DEBUG    | heisenberg_hamiltonians:__init__:482 - Symmetry group contains 0 elements
2023-10-18 16:49:13.571 | DEBUG    | heisenberg_hamiltonians:__init__:483 - Constructing basis


2023-10-18 16:49:13.616 | DEBUG    | heisenberg_hamiltonians:__init__:491 - Hilbert space dimension is 2704156
2023-10-18 16:49:13.627 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:69 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-1.0-False-None-10.pickle
2023-10-18 16:49:13.898 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:117 - Ground state energy is -43.0395889923
2023-10-18 16:49:13.899 | DEBUG    | __main__:<module>:21 - eps_train=0.01. Making train and test states...
2023-10-18 16:49:15.471 | INFO     | __main__:<module>:51 - Epoch 0, train loss: 0.6980
2023-10-18 16:49:15.680 | INFO     | __main__:<module>:54 - Overlap: 0.0043
2023-10-18 16:49:16.454 | INFO     | __main__:<module>:51 - Epoch 1, train loss: 0.6950
2023-10-18 16:49:16.629 | INFO     | __main__:<module>:54 - Overlap: -0.0014
2023-10-18 16:49:17.432 | INFO     | __main__:<module>:51 - Epoch 2, train loss: 0.6936
2023-10-18 16:49:17.607 

KeyboardInterrupt: 